# VaR と CVaR の基礎

このNotebookでは、損失分布から Value at Risk (VaR) と Conditional Value at Risk (CVaR) を計算します。

FinSimLabでは、**損失を正の値**として扱います。利益は負の損失です。

> 注意: このNotebookは教育目的です。投資助言や金融商品の推奨ではありません。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from finsimlab.risk import (
    calculate_returns,
    conditional_value_at_risk,
    losses_from_returns,
    plot_loss_distribution,
    value_at_risk,
    volatility,
)
from finsimlab.simulations import simulate_gbm

## 1. ヒストリカル法

まず、手元にある価格データからリターンを計算し、損失分布を作ります。ここでは小さなサンプルデータを使います。

In [ ]:
prices = np.array([100, 102, 101, 98, 99, 95, 97, 96, 100], dtype=float)
returns = calculate_returns(prices)
losses = losses_from_returns(returns, portfolio_value=10_000)

print("returns:", np.round(returns, 4))
print("losses:", np.round(losses, 2))
print("daily volatility:", round(volatility(returns), 4))

In [ ]:
var_95 = value_at_risk(losses, confidence_level=0.95)
cvar_95 = conditional_value_at_risk(losses, confidence_level=0.95)

print(f"95% VaR:  {var_95:,.2f}")
print(f"95% CVaR: {cvar_95:,.2f}")

plot_loss_distribution(losses, confidence_level=0.95, bins=8, title="Historical losses")
plt.show()

## 2. シミュレーション法

次に、幾何ブラウン運動で多数の将来価格パスを作り、1年後の損益分布からVaRとCVaRを計算します。

In [ ]:
paths = simulate_gbm(
    s0=100,
    mu=0.05,
    sigma=0.2,
    years=1,
    steps=252,
    n_paths=5_000,
    seed=42,
)

terminal_returns = paths[-1] / paths[0] - 1
simulated_losses = losses_from_returns(terminal_returns, portfolio_value=10_000)

for level in [0.90, 0.95, 0.99]:
    var_value = value_at_risk(simulated_losses, confidence_level=level)
    cvar_value = conditional_value_at_risk(simulated_losses, confidence_level=level)
    print(f"{level:.0%} VaR: {var_value:,.2f} / CVaR: {cvar_value:,.2f}")

In [ ]:
plot_loss_distribution(
    simulated_losses,
    confidence_level=0.95,
    bins=50,
    title="Simulated one-year losses",
)
plt.show()

## VaR と CVaR の違い

これらはどちらも「最悪の事態」を想定するための指標ですが、視点が異なります。

### 例え：テストの点数と補習
- **VaR (Value at Risk)**：**「下位5%の境界線」**です。「クラスで成績が悪い方から数えて5%目の人は何点か？」を知るようなものです。これによって、自分がどれくらい「最悪の事態」に近いかを確認できます。
- **CVaR (Conditional Value at Risk)**：**「赤点の人たちの平均点」**です。境界線（VaR）を越えてしまった人たちが、平均してどれくらいひどい点数だったかを示します。これにより、境界を越えた先の「本当の怖さ（テールリスク）」を把握できます。

### 実務上のポイント
銀行などの金融機関では、自己資本をいくら持っておくべきかを決める規制（バーゼル規制など）において、これらの指標が厳格に使われています。VaRは直感的で計算しやすい反面、CVaRは「起きてしまった時の深刻さ」をより正確に反映できるため、近年ではCVaRの重要性が高まっています。

## 学習メモ: 直感・数式・演習

### 直感

VaRは「ある信頼水準で、どの程度までの損失を見込むか」を表す指標です。FinSimLabでは、損失を正の値として扱います。たとえば95% VaRが `120` なら、この単純化された分布では「95%点の損失が120」という意味です。

CVaRは、VaRを超えるような悪いケースだけに注目した平均損失です。VaRは境界点を示しますが、その先の損失がどれほど大きいかは直接示しません。CVaRを見ると、tail riskの大きさを補足できます。

### 数式の最小説明

損失を $L$、信頼水準を $\alpha$ とすると、VaRは損失分布の $\alpha$ 分位点です。

$$\text{VaR}_\alpha = q_\alpha(L)$$

CVaRは、VaR以上の損失に注目した平均として直感的に理解できます。

$$\text{CVaR}_\alpha \approx E[L \mid L \geq \text{VaR}_\alpha]$$

FinSimLabの入門実装では、この符号規約を明確にするため、リターンを `losses_from_returns` で損失に変換してからVaR/CVaRを計算します。

### 演習問題

1. `confidence_level` を `0.90`, `0.95`, `0.99` に変えて、VaRとCVaRの変化を比較してください。
2. `portfolio_value` を大きくすると、損失額がどのようにスケールするか確認してください。
3. GBMシミュレーションの `sigma` を上げると、損失分布の右側の裾がどう変わるか観察してください。
4. VaRだけを見た場合とCVaRも見た場合で、リスクの印象がどう変わるかメモしてください。

> 注意: VaR/CVaRはリスクを要約する指標の一つです。将来の最大損失を保証するものではありません。
